# IndoBERT + SHAP: Indonesian Gov App Review Sentiment

**One-click experiment runner for Google Colab (GPU)**

This notebook clones the repository, installs dependencies, and runs the full 6-step pipeline:
1. Preprocessing
2. Baselines (TF-IDF + RF / LinearSVC)
3. IndoBERT fine-tuning
4. Evaluation & comparison
5. SHAP token attribution
6. Case studies & domain keyword validation

> **Runtime:** Go to `Runtime â†’ Change runtime type â†’ T4 GPU` before running.

## 0. Configuration

Set your repository URL and dataset options below.

In [ ]:
# ===== USER CONFIGURATION =====
REPO_URL = "https://github.com/ikhfa/igar-indobert-shap.git"

# Pipeline steps to run (None = all steps 1-6)
# Examples: "1" (preprocessing only), "1-3", "3" (IndoBERT only), None (all)
RUN_STEPS = None

## 1. GPU Check & Environment Setup

In [ ]:
import torch

# Verify GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected!")
    print("Go to Runtime -> Change runtime type -> T4 GPU")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 2. Clone Repository & Install Dependencies

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/igar-indobert-shap")

# Clone or pull
if PROJECT_DIR.exists():
    print("Repository exists, pulling latest...")
    !cd {PROJECT_DIR} && git pull
else:
    print(f"Cloning {REPO_URL} ...")
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install project dependencies (Colab already has torch, but we need the rest)
!pip install -q transformers>=4.35.0 datasets>=2.14.0 shap>=0.43.0 \
    scikit-learn>=1.3.0 pandas>=2.0.0 matplotlib>=3.7.0 seaborn>=0.12.0 \
    ipywidgets>=8.0.0 tqdm>=4.65.0 scipy>=1.11.0

print("Dependencies installed.")

## 3. Dataset Setup

Upload `Rating_labeled.csv` when prompted below.

In [ ]:
from google.colab import files
import config

data_dir = PROJECT_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)
csv_path = data_dir / "Rating_labeled.csv"

if not csv_path.exists():
    print("Upload Rating_labeled.csv:")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        with open(csv_path, "wb") as f:
            f.write(content)
        print(f"Saved {fname} -> {csv_path} ({len(content)/1e6:.1f} MB)")
else:
    print(f"Dataset already present: {csv_path}")

print(f"Dataset path: {config.DATASET_PATH}")

## 4. Run Full Pipeline

This runs all 6 steps of the experiment. On a T4 GPU, expect ~30-60 minutes with the full 617K dataset.

In [ ]:
step_arg = f"--step {RUN_STEPS}" if RUN_STEPS else ""
!cd {PROJECT_DIR} && python run_pipeline.py {step_arg}

## 5. View Results

In [ ]:
# List all generated output files
import subprocess
result = subprocess.run(["find", "output", "-type", "f"], capture_output=True, text=True, cwd=str(PROJECT_DIR))
print("Generated files:")
print(result.stdout)

In [ ]:
# Display model comparison table
import pandas as pd
from pathlib import Path

comparison_path = PROJECT_DIR / "output" / "metrics" / "comparison.csv"
if comparison_path.exists():
    df = pd.read_csv(comparison_path)
    print("=== Model Comparison ===")
    display(df)
else:
    print("comparison.csv not found. Run all pipeline steps first.")

In [ ]:
# Display key plots
from IPython.display import Image, display as ipy_display
from pathlib import Path

plots = [
    "cm_indobert.png",
    "model_comparison.png",
    "shap_top20_negative.png",
    "shap_top20_neutral.png",
    "shap_top20_positive.png",
    "class_distribution.png",
]

plots_dir = PROJECT_DIR / "output" / "plots"
for plot_name in plots:
    plot_path = plots_dir / plot_name
    if plot_path.exists():
        print(f"\n--- {plot_name} ---")
        ipy_display(Image(filename=str(plot_path), width=600))

In [ ]:
# Display training metrics log
metrics_path = PROJECT_DIR / "output" / "logs" / "training_metrics.csv"
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    print("=== Training History ===")
    display(metrics_df)

    # Plot training curves
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(metrics_df["epoch"], metrics_df["train_loss"], "b-o", label="Train")
    axes[0].plot(metrics_df["epoch"], metrics_df["val_loss"], "r-o", label="Val")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(metrics_df["epoch"], metrics_df["val_macro_f1"], "g-o", label="Val Macro F1")
    axes[1].plot(metrics_df["epoch"], metrics_df["val_accuracy"], "purple", marker="s", label="Val Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_title("Validation Metrics")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 6. Download Results

Download the full `output/` folder as a zip file.

In [ ]:
# Zip and download all outputs
!cd {PROJECT_DIR} && zip -r /content/igar_results.zip output/

from google.colab import files
files.download("/content/igar_results.zip")
print("Download started: igar_results.zip")

## Optional: Save to Google Drive

Mount Google Drive and copy results for persistent storage.

In [ ]:
# Uncomment to save results to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# 
# drive_dest = "/content/drive/MyDrive/igar-indobert-shap-results"
# !mkdir -p {drive_dest}
# !cp -r {PROJECT_DIR}/output/* {drive_dest}/
# !cp /content/igar_results.zip {drive_dest}/
# print(f"Results saved to Google Drive: {drive_dest}")